# Cleaner concept-bound initialization

The base concept-bound init (`concept_bound_vectors`, `class_weight_mode="uniform"`) mixes each
instance's **entire ancestor-materialized class hierarchy** into *every* class-bearing term:

- **own-class** mean (α),
- **outgoing** bindings `code(r) ⊙ code(c)` over `c ∈ cls*(o)` (β),
- **incoming** bindings `roll(code(r)) ⊙ code(c)` over `c ∈ cls*(s)` (γ).

Every ancestor enters with **equal weight** and is summed/accumulated, so deep hierarchies dump a lot
of generic-superclass mass into each vector (≈4.75 classes/instance on tc07). The bare-relation term
(δ) carries no class and is untouched everywhere.

This notebook tests two **less noisy** weightings of that hierarchy, both implemented as the new
`class_weight_mode` knob on `concept_bound_vectors` (see `scripts/_synthetic_compare.py`):

| variant | `class_weight_mode` | what it does |
|---|---|---|
| **(a) decay-weighted** | `"decay"` | keep ancestors, weight class `c` by `decay**hops` from the leaf (leaf=1, parent=0.5, …) so distant superclasses fade |
| **(b) most-specific** | `"most_specific"` | drop ancestors entirely; use only the instance's most-specific asserted class |

Both run on the **p2** and **p3** protographs. We compare against three baselines pulled from the
existing `synthetic_compare` run (identical config, so apples-to-apples): **vanilla**, **p2_bound**,
**p3_bound** (the uniform base). Metric: epoch-5 LogReg test accuracy on **tc07, tc09, tc14**.

> The base (`uniform`) path is byte-identical to before (verified `max|Δ|=0` over ~12k entities), so
> the cached `p2_bound`/`p3_bound` columns are directly reusable.


In [1]:
"""Setup: reuse synthetic_compare's walk cache + cached baselines."""
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
SCRIPTS = ROOT / "scripts"
# IMPORTANT: point at the existing synthetic_compare dir so cached walks/protographs are reused.
BASE_OUT = ROOT / "notebooks" / "synthetic_compare"
NEW_OUT = ROOT / "notebooks" / "clean_bound"
NEW_OUT.mkdir(parents=True, exist_ok=True)

if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from _synthetic_compare import ensure_walks, run_protograph_variant, write_protographs  # noqa: E402

TARGET_TCS = ["tc07", "tc09", "tc14"]

# mirror synthetic_compare.ipynb CFG exactly (so baselines are comparable)
CFG = dict(
    dim=200,
    walks_per_entity=100,
    proto_walks_per_entity=200,
    depth=3,
    epochs=5,
    pretrain_epochs=5,
    finetune_alpha=0.0025,
    min_alpha=0.0001,
    target_norm=8.0,
    seed=42,
    ancestor_decay=0.5,   # decay-weighted variant (a)
)


def tc_paths(tc: str) -> dict:
    tc_dir = ROOT / "v1" / "synthetic_ontology" / tc / "synthetic_ontology"
    return dict(
        ontology=tc_dir / "ontology.nt",
        graph=tc_dir / "graph.nt",
        train=tc_dir / "1000" / "train_test" / "train.txt",
        test=tc_dir / "1000" / "train_test" / "test.txt",
        out=BASE_OUT / tc,   # reuse cached walks here
    )


for tc in TARGET_TCS:
    for k, p in tc_paths(tc).items():
        if k != "out":
            assert p.is_file(), f"missing {k} for {tc}: {p}"
print("paths ok:", ", ".join(TARGET_TCS))


paths ok: tc07, tc09, tc14


In [2]:
"""Walk prep (cache hit against synthetic_compare): protographs + p2/p3 + instance walks."""
def prep_walks(tc: str) -> tuple[dict, Path]:
    p = tc_paths(tc)
    out = p["out"]
    out.mkdir(parents=True, exist_ok=True)
    proto_paths = write_protographs(p["ontology"], out)
    proto_walks = {
        kind: ensure_walks(
            proto_paths[kind],
            out / f"walks_{kind}.txt",
            walks_per_entity=CFG["proto_walks_per_entity"],
            depth=CFG["depth"],
            seed=CFG["seed"],
            ensure_triple_coverage=True,
        )
        for kind in ("p2", "p3")
    }
    inst_walks = ensure_walks(
        p["graph"],
        out / f"walks_instance_w{CFG['walks_per_entity']}_d{CFG['depth']}.txt",
        walks_per_entity=CFG["walks_per_entity"],
        depth=CFG["depth"],
        seed=CFG["seed"],
    )
    return proto_walks, inst_walks

print("prep_walks ready")


prep_walks ready


In [3]:
"""Run the 4 cleaner variants (cached to clean_bound/results.json; delete to re-run)."""
# (mode, name-suffix): variant (a) decay, variant (b) most-specific
NEW_VARIANTS = [("decay", "decay"), ("most_specific", "specific")]
RESULTS_JSON = NEW_OUT / "results.json"

proto_shared = dict(
    strategy="all_init",
    normalize=True,
    target_norm=CFG["target_norm"],
    finetune_alpha=CFG["finetune_alpha"],
    pretrain_epochs=CFG["pretrain_epochs"],
    dim=CFG["dim"],
    epochs=CFG["epochs"],
    min_alpha=CFG["min_alpha"],
    seed=CFG["seed"],
)


def load_cache() -> dict:
    return json.loads(RESULTS_JSON.read_text()) if RESULTS_JSON.is_file() else {}


def run_new_variants(tc: str, cache: dict, *, verbose: bool = True) -> list[dict]:
    if tc in cache:
        if verbose:
            print(f"{tc}: loaded {len(cache[tc])} cached variant(s)")
        return cache[tc]
    p = tc_paths(tc)
    proto_walks, inst_walks = prep_walks(tc)
    rows = []
    t0 = time.time()
    for kind in ("p2", "p3"):
        for mode, suffix in NEW_VARIANTS:
            name = f"{kind}_bound_{suffix}"
            res = run_protograph_variant(
                name, proto_walks[kind], inst_walks, p["ontology"], p["train"], p["test"],
                bound_graph=p["graph"],
                bound_class_weight_mode=mode,
                bound_ancestor_decay=CFG["ancestor_decay"],
                **proto_shared,
            )
            res.pop("model", None)
            res["tc"] = tc
            rows.append(res)
            if verbose:
                accs = " ".join(f"{a:.3f}" for a in res["accs"])
                print(f"  [{time.time()-t0:6.1f}s] {name:>18}  [{accs}]", flush=True)
    cache[tc] = rows
    RESULTS_JSON.write_text(json.dumps(cache, indent=2) + "\n")
    return rows

print("runner ready")


runner ready


In [4]:
"""Execute: reuse cached baselines, compute the 4 new variants per TC."""
base = json.loads((BASE_OUT / "results.json").read_text())
BASELINE_VARIANTS = ["vanilla", "p2_bound", "p3_bound"]

def baseline_acc(tc: str) -> dict:
    rows = {r["variant"]: r for r in base.get(tc, [])}
    return {v: rows[v]["final_acc"] for v in BASELINE_VARIANTS if v in rows}

cache = load_cache()
acc = {}
for tc in TARGET_TCS:
    print(f"=== {tc} ===")
    acc[tc] = dict(baseline_acc(tc))
    for r in run_new_variants(tc, cache):
        acc[tc][r["variant"]] = r["final_acc"]
print("done")


=== tc07 ===
tc07: loaded 4 cached variant(s)
=== tc09 ===
tc09: loaded 4 cached variant(s)
=== tc14 ===
tc14: loaded 4 cached variant(s)
done


In [5]:
"""Results: epoch-5 LogReg test accuracy. Columns = baselines + the 4 cleaner variants."""
COLS = [
    "vanilla",
    "p2_bound", "p3_bound",                 # uniform base (full hierarchy)
    "p2_bound_decay", "p3_bound_decay",     # (a) decay-weighted ancestors
    "p2_bound_specific", "p3_bound_specific",  # (b) most-specific only
]
df = pd.DataFrame({tc: {c: acc[tc].get(c, float("nan")) for c in COLS} for tc in TARGET_TCS}).T
df.index.name = "test case"
df.round(4)


,vanilla,p2_bound,p3_bound,p2_bound_decay,p3_bound_decay,p2_bound_specific,p3_bound_specific
test case,,,,,,,
tc07,0.5825,0.9925,0.9850,0.9550,0.9150,0.8350,0.8925
tc09,0.6350,0.8475,0.8025,0.8950,0.8225,0.8775,0.8075
tc14,0.6631,0.9050,0.9056,0.9581,0.9750,0.9812,0.9919


In [6]:
"""Improvement over vanilla (accuracy points)."""
delta = df.sub(df["vanilla"], axis=0).drop(columns=["vanilla"])
delta.round(4)


,p2_bound,p3_bound,p2_bound_decay,p3_bound_decay,p2_bound_specific,p3_bound_specific
test case,,,,,,
tc07,0.4100,0.4025,0.3725,0.3325,0.2525,0.3100
tc09,0.2125,0.1675,0.2600,0.1875,0.2425,0.1725
tc14,0.2419,0.2425,0.2950,0.3119,0.3181,0.3288


## Reading the table

Final epoch-5 accuracy (best per row in **bold**):

| test case | vanilla | p2_bound | p3_bound | p2_decay | p3_decay | p2_specific | p3_specific |
|---|---|---|---|---|---|---|---|
| **tc07** (∃P_19.C_390) | 0.583 | **0.993** | 0.985 | 0.955 | 0.915 | 0.835 | 0.893 |
| **tc09** (qualified cardinality) | 0.635 | 0.848 | 0.803 | **0.895** | 0.823 | 0.878 | 0.808 |
| **tc14** (counting / stability) | 0.663 | 0.905 | 0.906 | 0.958 | 0.975 | 0.981 | **0.992** |

**Every bound variant beats vanilla** by a wide margin (+0.17 to +0.41), so the transfer itself is
not in question — the variable is *how much class hierarchy* to inject.

**The optimal amount of hierarchy is task-dependent**, and it splits exactly along what makes each
task discriminative:

- **tc07 — keep the full hierarchy.** The label is "has a `P_19` edge to a `C_390`-thing", but the
  objects are typed with *deep leaf* classes that are only `C_390` *by inheritance*. The discriminative
  concept lives in the **ancestors**, so down-weighting them (decay: 0.955 / 0.915) or dropping them
  (most-specific: 0.835 / 0.893) strips the signal. The uniform base (**0.993**) is best precisely
  because of the "noise" we set out to remove.

- **tc09 / tc14 — prune the hierarchy.** These hinge on **edge counts**, not on a generic superclass.
  Here the ancestor mass is genuine noise diluting the cardinality magnitude, and cleaning helps:
  - tc09: `p2_decay` **0.895** > base `p2_bound` 0.848 (+0.047); `p2_specific` 0.878 also beats base.
  - tc14: cleaning is decisive — `p3_specific` **0.992** and `p3_decay` 0.975 vs base ≈0.906 (+0.07–0.09).

**Picking a default:** `decay` (0.5) is the safe middle ground — best or near-best on the counting
tasks while paying only a modest tax on tc07 (0.955 vs 0.993). `most_specific` is the strongest on
tc14 but the most damaging on tc07, so it's the right choice only when the task is known to key on
leaf type / cardinality rather than on an inherited superclass. **p2 vs p3** doesn't reorder the
story: the weighting scheme dominates the protograph choice on these three TCs.


# Does the cleaner init help on DBpedia (10% corpus)?

The same `class_weight_mode` knob is ported to DBpedia's **direction-aware** concept-bound init
([`scripts/_dbpedia_compare.concept_bound_vectors`](../scripts/_dbpedia_compare.py)). We train the two
cleaner variants (`decay`, `most_specific`) on **both p2 and p3**, finetuning on the **10% DBpedia
walk corpus** (`walks/all_walks_010.txt`, 13.4M walks). The init recipe is otherwise identical to the
DBpedia baselines (`target_norm=8`, `finetune_alpha=0.0025`, `rolled` direction tag, 5+5 epochs).

> **Baseline caveat — read before interpreting.** The `vanilla`, `p2_bound`, `p3_bound` columns are
> **reused from the full-corpus run** (`notebooks/dbpedia_compare`, 133M walks) — not retrained, per
> request. The 4 cleaner variants are trained on **10%** of that corpus (~10× less finetuning data).
> This is **not** a corpus-matched comparison: the cleaner methods are **data-handicapped** vs the
> baselines. Read it as *"does the cleaner hierarchy weighting help enough to show through even with
> 10× less finetuning data?"*, not as a head-to-head at equal data. A fully fair comparison would
> retrain p2_bound/p3_bound on 10% too.


In [7]:
"""DBpedia setup: 10% walk corpus, shared full inputs, output cache."""
from _dbpedia_compare import (  # noqa: E402
    corpus_vocab,
    load_eval_splits,
    load_instance_types,
    load_schema,
)
from _dbpedia_compare import ensure_walks as dbp_ensure_walks  # noqa: E402
from _dbpedia_compare import run_protograph_variant as dbp_run_variant  # noqa: E402
from _dbpedia_compare import write_protographs as dbp_write_protographs  # noqa: E402

DBP = dict(
    walks=ROOT / "walks" / "all_walks_010.txt",            # 10% instance-walk corpus
    ontology=ROOT / "dbpedia_graph" / "ontology.nt",
    graph=ROOT / "dbpedia_graph" / "graph.nt",
    dbpedia_root=ROOT / "v1" / "dbpedia",
    base_compare=ROOT / "notebooks" / "dbpedia_compare",   # full-corpus baselines + proto-walk cache
)
DBP_OUT = NEW_OUT / "dbpedia"                              # notebooks/clean_bound/dbpedia
DBP_OUT.mkdir(parents=True, exist_ok=True)
DBP_RESULTS = DBP_OUT / "results.json"

DCFG = dict(
    dim=200, epochs=5, pretrain_epochs=5,
    finetune_alpha=0.0025, min_alpha=0.0001, target_norm=8.0,
    direction_tag="rolled", ancestor_decay=0.5,
    proto_walks_per_entity=200, proto_depth=3,
    eval_k=5000, seed=42, workers=20,
)
# (protograph kind, class_weight_mode, name suffix)
DBP_NEW = [
    ("p2", "decay", "decay"), ("p3", "decay", "decay"),
    ("p2", "most_specific", "specific"), ("p3", "most_specific", "specific"),
]
DBP_NEW_NAMES = [f"{k}_bound_{sfx}" for k, _m, sfx in DBP_NEW]

for p in (DBP["walks"], DBP["ontology"], DBP["graph"]):
    assert p.is_file(), f"missing {p}"
assert (DBP["base_compare"] / "results.json").is_file(), "full-corpus baselines not found"
print("DBpedia inputs ok. New 10% variants:", DBP_NEW_NAMES)


DBpedia inputs ok. New 10% variants: ['p2_bound_decay', 'p3_bound_decay', 'p2_bound_specific', 'p3_bound_specific']


In [8]:
"""Train the 4 cleaner variants on the 10% corpus (cache-aware; per-variant JSON).

HEAVY: each variant builds bound vectors over the full graph and finetunes on 13.4M walks.
Cached rows are skipped, so re-executing this notebook is cheap once results.json is populated.
Delete notebooks/clean_bound/dbpedia/results.json to force a re-run.
"""
dbp_results = json.loads(DBP_RESULTS.read_text()) if DBP_RESULTS.is_file() else {}
todo = [(k, m, nm) for (k, m, _sfx), nm in zip(DBP_NEW, DBP_NEW_NAMES) if nm not in dbp_results]
print(f"cached: {sorted(dbp_results)}\nto run: {[t[2] for t in todo]}", flush=True)

if todo:
    schema = load_schema(DBP["ontology"])
    print("schema:", schema.stats, flush=True)
    # reuse the cached full-corpus protograph walks (write_protographs is idempotent)
    proto_paths = dbp_write_protographs(schema, DBP["base_compare"] / "protographs")
    proto_walks = {
        kind: dbp_ensure_walks(
            proto_paths[kind], DBP["base_compare"] / "protographs" / f"walks_{kind}.txt",
            walks_per_entity=DCFG["proto_walks_per_entity"], depth=DCFG["proto_depth"], seed=DCFG["seed"],
        )
        for kind in {t[0] for t in todo}
    }
    print("loading instance types from graph.nt ...", flush=True)
    itypes = load_instance_types(DBP["graph"], schema)
    print("itypes:", itypes.stats, flush=True)
    vocab_cache = DBP["walks"].with_name(DBP["walks"].stem + "_vocab.pkl")
    freq, n_lines, n_tokens = corpus_vocab(DBP["walks"], vocab_cache)
    print(f"corpus: {n_lines} walks, {n_tokens} tokens, {len(freq)} vocab", flush=True)
    splits = load_eval_splits(DBP["dbpedia_root"], k=DCFG["eval_k"])
    print(f"eval splits: {len(splits)}", flush=True)

    for kind, mode, name in todo:
        print(f"=== {name}  (kind={kind}, class_weight_mode={mode}) ===", flush=True)
        res = dbp_run_variant(
            name, proto_walks[kind], DBP["walks"], DBP["graph"],
            freq, n_lines, n_tokens, splits, itypes, schema,
            bound=True,
            bound_class_weight_mode=mode, bound_ancestor_decay=DCFG["ancestor_decay"],
            dim=DCFG["dim"], epochs=DCFG["epochs"], pretrain_epochs=DCFG["pretrain_epochs"],
            finetune_alpha=DCFG["finetune_alpha"], min_alpha=DCFG["min_alpha"],
            target_norm=DCFG["target_norm"], direction_tag=DCFG["direction_tag"],
            seed=DCFG["seed"], workers=DCFG["workers"], save_kv=None,
        )
        res["corpus"], res["class_weight_mode"] = "all_walks_010.txt", mode
        dbp_results[name] = res
        DBP_RESULTS.write_text(json.dumps(dbp_results, indent=2) + "\n")
        print(f"  {name}: init {res['init_mean_acc']:.4f} -> final {res['final_mean_acc']:.4f} "
              f"({res['seconds']:.0f}s)", flush=True)
print("dbpedia variant runs ready")


cached: []
to run: ['p2_bound_decay', 'p3_bound_decay', 'p2_bound_specific', 'p3_bound_specific']


schema: {'ontology_iri_triples': 18649, 'declared_classes': 790, 'subclass_edges_raw': 813, 'subclass_edges_kept': 724, 'props_with_domain_raw': 2499, 'props_with_range_raw': 2735, 'props_with_domain_and_range_raw': 2286, 'props_with_declared_domain_and_range': 700}


loading instance types from graph.nt ...


itypes: {'type_rows_total': 7335331, 'type_rows_declared_class': 6613247, 'typed_entities': 6562746, 'classes_used': 423}


corpus: 13350182 walks, 116138294 tokens, 928518 vocab


...

/home/bnukhkadiev/Knowledge-Graphs/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


    [    762s] p3_bound_specific epoch 5: mean acc 0.8191 (eval 38s)


/home/bnukhkadiev/Knowledge-Graphs/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  p3_bound_specific: init 0.7791 -> final 0.8191 (902s)


dbpedia variant runs ready


In [9]:
"""DBpedia comparison. Baselines (vanilla/p2_bound/p3_bound) = FULL corpus (reused, not retrained);
the 4 cleaner variants = 10% corpus. Metric: mean LogReg test acc over DLCC splits, epoch-5."""
from collections import defaultdict
from IPython.display import display

base_dbp = json.loads((DBP["base_compare"] / "results.json").read_text())
dbp_results = json.loads(DBP_RESULTS.read_text()) if DBP_RESULTS.is_file() else {}


def _final_split_accs(row):
    return {s: accs[-1] for s, accs in row["split_accs"].items()}


def _per_tc_means(row):
    g = defaultdict(list)
    for s, a in _final_split_accs(row).items():
        g[s.split("/")[0]].append(a)
    return {tc: float(np.mean(v)) for tc, v in g.items()}


# (column label, store, key, corpus tag)
DBP_COLS = [
    ("vanilla", base_dbp, "vanilla", "full"),
    ("p2_bound", base_dbp, "p2_bound", "full"),
    ("p3_bound", base_dbp, "p3_bound", "full"),
    ("p2_bound_decay", dbp_results, "p2_bound_decay", "10%"),
    ("p3_bound_decay", dbp_results, "p3_bound_decay", "10%"),
    ("p2_bound_specific", dbp_results, "p2_bound_specific", "10%"),
    ("p3_bound_specific", dbp_results, "p3_bound_specific", "10%"),
]
summary, per_tc = {}, {}
for col, store, key, corpus in DBP_COLS:
    label = f"{col} ({corpus})"
    if key in store:
        summary[label] = store[key]["final_mean_acc"]
        per_tc[label] = _per_tc_means(store[key])
    else:
        summary[label] = float("nan")
        per_tc[label] = {}

df_dbp = pd.DataFrame({"mean acc (all DLCC splits)": summary}).T
print("Headline — mean LogReg test accuracy over all DLCC DBpedia splits "
      "(baselines FULL corpus, cleaner variants 10%):")
display(df_dbp.round(4))
print("\nPer-test-case mean (final epoch):")
df_dbp_tc = pd.DataFrame(per_tc).sort_index()
df_dbp_tc.index.name = "test case"
df_dbp_tc.round(4)


Headline — mean LogReg test accuracy over all DLCC DBpedia splits (baselines FULL corpus, cleaner variants 10%):


,vanilla (full),p2_bound (full),p3_bound (full),p2_bound_decay (10%),p3_bound_decay (10%),p2_bound_specific (10%),p3_bound_specific (10%)
mean acc (all DLCC splits),0.8309,0.8028,0.7975,0.8215,0.8131,0.8279,0.8191



Per-test-case mean (final epoch):


,vanilla (full),p2_bound (full),p3_bound (full),p2_bound_decay (10%),p3_bound_decay (10%),p2_bound_specific (10%),p3_bound_specific (10%)
test case,,,,,,,
tc01,0.8456,0.8727,0.8691,0.8851,0.8767,0.8901,0.8751
tc02,0.7953,0.8183,0.8210,0.7757,0.7597,0.8287,0.8299
tc03,0.9263,0.8995,0.8893,0.8816,0.9021,0.8986,0.9008
tc04,0.9097,0.7911,0.7984,0.8267,0.8261,0.8176,0.8187
tc05,0.9693,0.7645,0.8099,0.8846,0.8798,0.8691,0.8660
tc06,0.8982,0.8089,0.8008,0.8309,0.8132,0.8200,0.8076
tc07,0.9512,0.9678,0.9627,0.9853,0.9840,0.9890,0.9881
tc08,0.9025,0.8739,0.8642,0.8739,0.8676,0.8826,0.8808
tc09,0.8267,0.8672,0.8578,0.8923,0.8807,0.8897,0.8878


## Reading the DBpedia table

Final-epoch LogReg test accuracy, split into **normal** (58 splits) vs **hard** (31 splits, hard
negatives) and overall. **Baselines are full corpus; the 4 cleaner variants are 10%** — so the
cleaner methods carry a ~10× data handicap (read the *hard* column and the *vs-uniform-bound*
comparison, not the raw "all" gap to vanilla).

| variant | corpus | normal | hard | all |
|---|---|---|---|---|
| vanilla | full | **0.9012** | 0.6995 | 0.8309 |
| p2_bound (uniform) | full | 0.8503 | 0.7141 | 0.8028 |
| p3_bound (uniform) | full | 0.8515 | 0.6963 | 0.7975 |
| p2_bound_decay | 10% | 0.8701 | **0.7307** | 0.8215 |
| p3_bound_decay | 10% | 0.8701 | 0.7066 | 0.8131 |
| **p2_bound_specific** | 10% | 0.8815 | 0.7276 | **0.8279** |
| p3_bound_specific | 10% | 0.8674 | 0.7289 | 0.8191 |

**1. The cleaner inits beat the uniform bound — on a 10× smaller corpus.** Every one of the four 10%
cleaner variants beats *both* full-corpus uniform-bound baselines on overall accuracy (0.819–0.828 vs
p2_bound 0.803 / p3_bound 0.798), and on the hard splits too. So the win isn't a data artifact — the
cleaner methods overcome a 10× data disadvantage relative to the uniform bound. This is the DBpedia
echo of the synthetic counting-TC finding: trimming generic-superclass noise sharpens the signal.

**2. The bound advantage lives in the hard splits.** On *normal* splits full-corpus vanilla wins
(0.901) — easy random-negative tasks where from-scratch RDF2Vec on the full corpus is already strong.
On *hard* splits the order flips: all four cleaner variants beat vanilla (0.6995), led by
`p2_bound_decay` at **0.7307 (+3.1 pts)**. Hard negatives are where the relational / concept-binding
structure the init injects actually pays off.

**3. Cleaning helps on *both* axes vs the uniform bound.** Δ vs full vanilla, uniform → cleaner:
normal deficit shrinks (`p2`: −0.051 → −0.020 for `specific`) **and** hard gain grows
(`p2`: +0.015 → +0.031 for `decay`). So the hierarchy-noise reduction recovers easy-split accuracy
*and* widens the hard-split edge at the same time.

**4. decay vs most_specific.** `most_specific` is the best overall and on normal splits
(`p2_bound_specific` 0.8279 / 0.8815); `decay` edges it on hard (`p2_bound_decay` 0.7307). On DBpedia,
**pruning the hierarchy hardest (`most_specific`) is the safest default** — the opposite of synthetic
tc07, where the discriminative class *was* a superclass so pruning hurt. Consistent reading: keep the
hierarchy only when the task keys on an inherited superclass; DBpedia's DLCC tasks key on the specific
type + relations, so the generic ancestor mass is mostly noise here.

**Caveat / next step.** Because the baselines are full-corpus, the "all" column understates the
cleaner methods. A corpus-matched run (uniform `p2_bound`/`p3_bound` retrained on 10%, or the cleaner
variants on full corpus) would convert this from "overcomes the handicap" to a clean head-to-head.
